# Data Treatment — Cleaning and Preparing Drill Hole Data

**Objective:** Apply systematic data cleaning to the assay database, preparing it for block model estimation.

**Steps:**
1. Replace sentinel values (-99) with NaN
2. Fix swapped coordinates in the collar table
3. Compute 3D sample center coordinates using azimuth and dip vectors
4. Calculate sample lengths
5. Validate and export cleaned dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = os.path.join("..", "data")

## 1. Load Raw Data

In [ ]:
assays = pd.read_csv(os.path.join(DATA_DIR, "assays.csv"))
collar = pd.read_csv(os.path.join(DATA_DIR, "collar.csv"))
block_model = pd.read_csv(os.path.join(DATA_DIR, "block_model.csv"))

print(f"Assays:      {assays.shape}")
print(f"Collar:      {collar.shape}")
print(f"Block Model: {block_model.shape}")

## 2. Replace Sentinel Values (-99 → NaN)

The value `-99` is used as a placeholder for missing geochemical analyses. We replace them with `NaN` so they are properly excluded from statistical calculations.

In [ ]:
geochem_cols = ["FE", "SI", "G1", "G2", "G3"]

# Before cleaning
print("Before cleaning — samples with -99:")
for col in geochem_cols:
    count = (assays[col] == -99).sum()
    print(f"  {col}: {count:>5} ({count/len(assays)*100:.1f}%)")

# Replace -99 with NaN
for col in geochem_cols:
    assays[col] = assays[col].replace(-99, np.nan)

# Also clean block model
for col in geochem_cols:
    block_model[col] = block_model[col].replace(-99, np.nan)

print("\nAfter cleaning — NaN counts:")
print(assays[geochem_cols].isna().sum())

## 3. Fix Swapped Coordinates

During EDA, we identified that some collar entries have X and Y coordinates swapped. The expected coordinate ranges for this area (UTM Zone 23S) are:
- **Easting (X):** ~640,000 – 642,000
- **Northing (Y):** ~8,425,000 – 8,429,000

Rows with X > 1,000,000 clearly have the columns swapped.

In [ ]:
# Identify and fix swapped coordinates in the collar table
swap_mask = collar["X"] > 1_000_000
print(f"Collar rows with swapped X/Y: {swap_mask.sum()}")
print("\nBefore fix:")
print(collar.loc[swap_mask, ["FURO", "X", "Y", "Z"]].head())

collar.loc[swap_mask, ["X", "Y"]] = collar.loc[swap_mask, ["Y", "X"]].values

print("\nAfter fix:")
print(collar.loc[swap_mask, ["FURO", "X", "Y", "Z"]].head())

## 4. Compute 3D Sample Center Coordinates

Each assay interval is defined by a collar position plus downhole survey data (azimuth and dip). To locate the sample in 3D space, we use direction cosines to project the interval start (`DE`) and end (`ATE`) positions along the borehole trajectory, then compute the midpoint.

**Direction cosines:**
- `dx = sin(azimuth) × cos(dip)`
- `dy = cos(azimuth) × cos(dip)`
- `dz = -sin(dip)`

In [ ]:
# Convert azimuth and dip from degrees to radians
az_rad = np.radians(assays["AZ"].fillna(0))
dip_rad = np.radians(assays["DIP"].fillna(0))

# Direction cosines
dx = np.sin(az_rad) * np.cos(dip_rad)
dy = np.cos(az_rad) * np.cos(dip_rad)
dz = -np.sin(dip_rad)

# Interval start point (projected from collar)
assays["X_start"] = assays["XCOLLAR"] + assays["DE"] * dx
assays["Y_start"] = assays["YCOLLAR"] + assays["DE"] * dy
assays["Z_start"] = assays["ZCOLLAR"] + assays["DE"] * dz

# Interval end point
assays["X_end"] = assays["XCOLLAR"] + assays["ATE"] * dx
assays["Y_end"] = assays["YCOLLAR"] + assays["ATE"] * dy
assays["Z_end"] = assays["ZCOLLAR"] + assays["ATE"] * dz

# Midpoint (sample center)
assays["X_center"] = (assays["X_start"] + assays["X_end"]) / 2
assays["Y_center"] = (assays["Y_start"] + assays["Y_end"]) / 2
assays["Z_center"] = (assays["Z_start"] + assays["Z_end"]) / 2

print("Sample center coordinates computed successfully.")
assays[["FURO", "AMOSTRA", "X_center", "Y_center", "Z_center"]].head(10)

## 5. Calculate Sample Length

The sample length (`ATE - DE`) is used as a weight in the block estimation step — longer intervals contribute more to the weighted average.

In [ ]:
assays["LENGTH"] = assays["ATE"] - assays["DE"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(assays["LENGTH"], bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("Sample Length (m)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Sample Interval Lengths")
ax.axvline(assays["LENGTH"].mean(), color="black", linestyle="--",
           label=f'Mean = {assays["LENGTH"].mean():.2f} m')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Length range: {assays['LENGTH'].min():.2f} – {assays['LENGTH'].max():.2f} m")
print(f"Mean length: {assays['LENGTH'].mean():.2f} m")
print(f"Negative or zero lengths: {(assays['LENGTH'] <= 0).sum()}")

## 6. Validation — 3D Sample Positions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plan view of sample centers colored by Fe
valid = assays.dropna(subset=["FE"])
sc = axes[0].scatter(valid["X_center"], valid["Y_center"], c=valid["FE"],
                     cmap="RdYlGn", s=3, alpha=0.5)
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")
axes[0].set_title("Sample Centers — Plan View (colored by Fe)")
axes[0].ticklabel_format(style="plain")
plt.colorbar(sc, ax=axes[0], label="Fe (%)")

# Cross-section (X vs Z)
sc2 = axes[1].scatter(valid["X_center"], valid["Z_center"], c=valid["FE"],
                      cmap="RdYlGn", s=3, alpha=0.5)
axes[1].set_xlabel("Easting (m)")
axes[1].set_ylabel("Elevation (m)")
axes[1].set_title("Sample Centers — Cross Section (colored by Fe)")
plt.colorbar(sc2, ax=axes[1], label="Fe (%)")

plt.tight_layout()
plt.show()

## 7. Cleaned Data Summary

In [ ]:
print("=" * 60)
print("CLEANED DATA SUMMARY")
print("=" * 60)
print(f"\nTotal samples: {len(assays):,}")
print(f"Unique drill holes: {assays['FURO'].nunique()}")
print(f"Lithotype categories: {assays['Lito_Final'].nunique()}")

print("\nValid (non-NaN) sample counts per variable:")
for col in geochem_cols:
    valid_count = assays[col].notna().sum()
    print(f"  {col}: {valid_count:>5} ({valid_count/len(assays)*100:.1f}%)")

print(f"\nSample length — mean: {assays['LENGTH'].mean():.2f} m, "
      f"median: {assays['LENGTH'].median():.2f} m")

print("\nDescriptive statistics (cleaned):")
assays[geochem_cols].describe().round(2)

## 8. Export Cleaned Data

Save the treated assays and corrected collar tables for use in the block model estimation step.

In [ ]:
# Export cleaned datasets
assays.to_csv(os.path.join(DATA_DIR, "assays_cleaned.csv"), index=False)
collar.to_csv(os.path.join(DATA_DIR, "collar_cleaned.csv"), index=False)
block_model.to_csv(os.path.join(DATA_DIR, "block_model_cleaned.csv"), index=False)

print("Exported files:")
print("  data/assays_cleaned.csv")
print("  data/collar_cleaned.csv")
print("  data/block_model_cleaned.csv")

## Summary

**Treatments applied:**

| Step | Action | Impact |
|------|--------|--------|
| Sentinel replacement | -99 → NaN | Prevents bias in statistical calculations |
| Coordinate fix | Swapped X/Y corrected in collar table | Ensures accurate spatial positioning |
| 3D positioning | Computed sample center coordinates via direction cosines | Required for block assignment |
| Sample length | Calculated interval length (ATE - DE) | Used as weight in grade estimation |

**Next step:** Block model estimation — assign samples to blocks and compute length-weighted average grades.